In [6]:
# Testing section-based chunking for fine-tuning
import sys
sys.path.append('../..')
import utils.llm_training as llm_training
from transformers import AutoTokenizer
import re

# Load tokenizer for testing
tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B", trust_remote_code=True)

def chunk_text_by_subsections(text_content: str, tokenizer, max_tokens: int = 2048):
    """
    Helper function to chunk a section by subsections when possible.
    
    Args:
        text_content: The section text to chunk
        tokenizer: The tokenizer to use for counting tokens
        max_tokens: Maximum tokens per chunk
        
    Returns:
        List of text chunks and total token count
    """
    subsection_pattern = r'(\\subsection\{[^}]+\})'
    parts = re.split(subsection_pattern, text_content)
    
    subsections = []
    
    # Handle content before first subsection
    if parts[0].strip():
        subsections.append(parts[0])
    
    # Group subsection headers with their content
    i = 1
    while i < len(parts):
        if re.match(subsection_pattern, parts[i]):  # This is a subsection header
            subsection_content = parts[i]  # Start with the subsection header
            if i + 1 < len(parts):
                subsection_content += parts[i + 1]  # Add the content after the header
            subsections.append(subsection_content)
            i += 2
        else:
            i += 1
    
    # If we only have one subsection (no actual subsection splits), return original
    if len(subsections) <= 1:
        return [text_content], len(tokenizer(text_content, add_special_tokens=False, truncation=False)["input_ids"])
    
    chunks = []
    total_tokens = 0
    current_chunk = ""
    
    for subsection in subsections:
        if not subsection.strip():
            continue
            
        # Check if this would exceed max_tokens when added to current chunk
        test_chunk = current_chunk + subsection
        tokens = tokenizer(test_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        token_count = len(tokens)
        
        if token_count <= max_tokens:
            # Add to current chunk
            current_chunk = test_chunk
        else:
            # Save current chunk if it has content
            if current_chunk.strip():
                chunks.append(current_chunk)
                chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
                total_tokens += len(chunk_tokens)
            
            # Check if this subsection alone exceeds max_tokens
            subsection_tokens = tokenizer(subsection, add_special_tokens=False, truncation=False)["input_ids"]
            if len(subsection_tokens) > max_tokens:
                print(f"Subsection too large ({len(subsection_tokens)} tokens), using token-based chunking...")
                # Fall back to token-based chunking for this subsection
                subsection_chunks, subsection_token_count = llm_training.chunk_text(subsection, tokenizer, max_tokens)
                chunks.extend(subsection_chunks)
                total_tokens += subsection_token_count
                current_chunk = ""
            else:
                # Start new chunk with this subsection
                current_chunk = subsection
    
    # Add final chunk if it has content
    if current_chunk.strip():
        chunks.append(current_chunk)
        chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        total_tokens += len(chunk_tokens)
    
    return chunks, total_tokens

def chunk_text_by_sections(text_content: str, tokenizer, max_tokens: int = 2048):
    """
    Chunks text by sections, ensuring each chunk doesn't exceed max_tokens.
    If a section is too large, it tries to split by subsections first.
    If no subsections exist or they're still too large, it falls back to token-based chunking.
    
    Args:
        text_content: The full text to chunk
        tokenizer: The tokenizer to use for counting tokens
        max_tokens: Maximum tokens per chunk
        
    Returns:
        List of text chunks and total token count
    """
    # Find all section boundaries and split into sections
    section_pattern = r'(\\section\{[^}]+\})'
    parts = re.split(section_pattern, text_content)
    
    # Group content: first part is pre-section content, then pairs of (section_header, section_content)
    sections = []
    
    # Handle content before first section (title, abstract, etc.)
    if parts[0].strip():
        sections.append(parts[0])
    
    # Group section headers with their content
    i = 1
    while i < len(parts):
        if re.match(section_pattern, parts[i]):  # This is a section header
            section_content = parts[i]  # Start with the section header
            if i + 1 < len(parts):
                section_content += parts[i + 1]  # Add the content after the header
            sections.append(section_content)
            i += 2
        else:
            i += 1
    
    chunks = []
    total_tokens = 0
    current_chunk = ""
    
    for section in sections:
        if not section.strip():
            continue
            
        # Check if this would exceed max_tokens when added to current chunk
        test_chunk = current_chunk + section
        tokens = tokenizer(test_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        token_count = len(tokens)
        
        if token_count <= max_tokens:
            # Add to current chunk
            current_chunk = test_chunk
        else:
            # Save current chunk if it has content
            if current_chunk.strip():
                chunks.append(current_chunk)
                chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
                total_tokens += len(chunk_tokens)
            
            # Check if this section alone exceeds max_tokens
            section_tokens = tokenizer(section, add_special_tokens=False, truncation=False)["input_ids"]
            if len(section_tokens) > max_tokens:
                print(f"Section too large ({len(section_tokens)} tokens), trying subsection chunking...")
                # Try to split by subsections first
                subsection_chunks, subsection_token_count = chunk_text_by_subsections(section, tokenizer, max_tokens)
                chunks.extend(subsection_chunks)
                total_tokens += subsection_token_count
                current_chunk = ""
            else:
                # Start new chunk with this section
                current_chunk = section
    
    # Add final chunk if it has content
    if current_chunk.strip():
        chunks.append(current_chunk)
        chunk_tokens = tokenizer(current_chunk, add_special_tokens=False, truncation=False)["input_ids"]
        total_tokens += len(chunk_tokens)
    
    return chunks, total_tokens


In [4]:
# Test with the DPO paper
with open('../../data/arxiv/cleaned_DPO.txt', 'r', encoding='utf-8') as f:
    dpo_paper = f.read()

print("Testing section-based chunking...")
section_chunks, section_total_tokens = chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048)

print(f"Section-based chunking: {len(section_chunks)} chunks, {section_total_tokens} total tokens")

# Compare with original token-based chunking
token_chunks, token_total_tokens = llm_training.chunk_text(dpo_paper, tokenizer, 2048)
print(f"Token-based chunking: {len(token_chunks)} chunks, {token_total_tokens} total tokens")

print(f"\nDifference: {len(section_chunks) - len(token_chunks)} chunks")

# Let's examine the chunk boundaries
print("\n=== SECTION-BASED CHUNK BOUNDARIES ===")
for i, chunk in enumerate(section_chunks):
    first_line = chunk.strip().split('\n')[0][:100] + "..." if len(chunk.strip()) > 100 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")

print("\n=== TOKEN-BASED CHUNK BOUNDARIES (first 3) ===")
for i, chunk in enumerate(token_chunks[:3]):
    first_line = chunk.strip().split('\n')[0][:100] + "..." if len(chunk.strip()) > 100 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")


Testing section-based chunking...
Section too large (2086 tokens), using token-based chunking...
Section too large (4171 tokens), using token-based chunking...
Section-based chunking: 10 chunks, 11928 total tokens
Token-based chunking: 6 chunks, 11928 total tokens

Difference: 4 chunks

=== SECTION-BASED CHUNK BOUNDARIES ===
Chunk 1: 1334 tokens - '\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}...'
Chunk 2: 904 tokens - '\section{Related Work}...'
Chunk 3: 1183 tokens - '\section{Preliminaries}\label{section:prelims}...'
Chunk 4: 2048 tokens - '\section{Direct Preference Optimization}\label{sec:DPO}...'
Chunk 5: 38 tokens - 'distribution which is unavailable, and $\piref$ used by DPO. Further details related to the implemen...'
Chunk 6: 1855 tokens - '\section{Theoretical Analysis of DPO}...'
Chunk 7: 2048 tokens - '\section{Experiments}...'
Chunk 8: 2048 tokens - 'under the true reward function as well as the average sequence-level KL\footnote{T

In [8]:
# Test the improved function with subsection handling
print("Testing IMPROVED section+subsection-based chunking...")
improved_chunks, improved_total_tokens = chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048)

print(f"Section+subsection chunking: {len(improved_chunks)} chunks, {improved_total_tokens} total tokens")
print(f"Original token-based chunking: {len(token_chunks)} chunks, {token_total_tokens} total tokens")
print(f"Difference: {len(improved_chunks) - len(token_chunks)} chunks")

print("\n=== IMPROVED SECTION+SUBSECTION CHUNK BOUNDARIES ===")
for i, chunk in enumerate(improved_chunks):
    first_line = chunk.strip().split('\n')[0][:100] + "..." if len(chunk.strip()) > 100 else chunk.strip().split('\n')[0]
    tokens = tokenizer(chunk, add_special_tokens=False, truncation=False)["input_ids"]
    print(f"Chunk {i+1}: {len(tokens)} tokens - '{first_line}'")

# Let's check if we can find any subsections in the text to see if they're being handled
print("\n=== CHECKING FOR SUBSECTIONS IN DPO PAPER ===")
subsection_matches = re.findall(r'\\subsection\{[^}]+\}', dpo_paper)
print(f"Found {len(subsection_matches)} subsections:")
for match in subsection_matches[:5]:  # Show first 5
    print(f"  {match}")

Testing IMPROVED section+subsection-based chunking...
Section too large (2086 tokens), trying subsection chunking...
Section too large (4171 tokens), trying subsection chunking...
Section+subsection chunking: 9 chunks, 11928 total tokens
Original token-based chunking: 6 chunks, 11928 total tokens
Difference: 3 chunks

=== IMPROVED SECTION+SUBSECTION CHUNK BOUNDARIES ===
Chunk 1: 1334 tokens - '\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}...'
Chunk 2: 904 tokens - '\section{Related Work}...'
Chunk 3: 1183 tokens - '\section{Preliminaries}\label{section:prelims}...'
Chunk 4: 2086 tokens - '\section{Direct Preference Optimization}\label{sec:DPO}...'
Chunk 5: 1855 tokens - '\section{Theoretical Analysis of DPO}...'
Chunk 6: 1644 tokens - '\section{Experiments}...'
Chunk 7: 1799 tokens - '\subsection{How well can DPO optimize the RLHF objective?}...'
Chunk 8: 728 tokens - '\subsection{Validating GPT-4 judgments with human judgments}...'
Chunk 9: 395

In [9]:
# Test the integration with fine_tune_on_text function
print("Testing integration with fine_tune_on_text...")

# Import the updated function
import importlib
importlib.reload(llm_training)

# Test that the function signature includes the new parameter
import inspect
sig = inspect.signature(llm_training.fine_tune_on_text)
print(f"fine_tune_on_text parameters: {list(sig.parameters.keys())}")

# Check if chunk_by_section parameter exists and has correct default
chunk_by_section_param = sig.parameters.get('chunk_by_section')
if chunk_by_section_param:
    print(f"chunk_by_section parameter found with default: {chunk_by_section_param.default}")
else:
    print("ERROR: chunk_by_section parameter not found!")

# Test the chunking functions directly
print("\n=== Testing chunking functions directly ===")

# Test section-based chunking function
test_chunks, test_tokens = llm_training.chunk_text_by_sections(dpo_paper, tokenizer, max_tokens=2048)
print(f"Section-based: {len(test_chunks)} chunks, {test_tokens} tokens")

# Test original chunking function  
orig_chunks, orig_tokens = llm_training.chunk_text(dpo_paper, tokenizer, 2048)
print(f"Token-based: {len(orig_chunks)} chunks, {orig_tokens} tokens")

print(f"Functions working correctly: {test_tokens == orig_tokens}")  # Should have same total tokens


Testing integration with fine_tune_on_text...
fine_tune_on_text parameters: ['model', 'tokenizer', 'log', 'text_content', 'train_cfg', 'train', 'tag', 'callbacks', 'chunk_by_section']
chunk_by_section parameter found with default: False

=== Testing chunking functions directly ===
Section-based: 9 chunks, 11928 tokens
Token-based: 6 chunks, 11928 tokens
Functions working correctly: True


In [ ]:
# Example of how to use the new functionality
print("=== USAGE EXAMPLES ===")

print("\n1. To use section-based chunking in your fine-tuning script:")
print("   python scripts/FT/finetuning_knowledge_v5.py --chunk_by_section")

print("\n2. To use token-based chunking (default):")
print("   python scripts/FT/finetuning_knowledge_v5.py")

print("\n3. In code, you can now call:")
print("   llm_training.fine_tune_on_text(..., chunk_by_section=True)")

print("\n=== EXPECTED BENEFITS ===")
print("✓ Better semantic boundaries - chunks respect paper structure")
print("✓ Sections stay together - no cutting in the middle of important content")
print("✓ Includes section/subsection headers for context")
print("✓ Falls back to token-based chunking for oversized sections")
print("✓ Handles content before first section (title, abstract)")

print(f"\n=== CHUNKING COMPARISON FOR DPO PAPER ===")
print(f"Section-based: {len(improved_chunks)} chunks")
print(f"Token-based:   {len(token_chunks)} chunks")
print(f"More chunks with section-based = More respect for semantic boundaries")
